# Step 3 — MTP Head INT8 Quantization


In [ ]:
# ── Paths & constants 

import os
import torch
from safetensors import safe_open
from safetensors.torch import save_file
from compressed_tensors.compressors import pack_to_int32

CKPT  = "/home/prashant.takale/icml/4_models/qwen3.5-4b-w8a16-clean/model.safetensors"
GROUP = 128
NB    = 8      # INT8
QMAX  = 2 ** (NB - 1) - 1  # 127

# The 7 MTP Linear modules — self_attn projections + MLP gate/up/down
# mtp.fc and norms are deliberately left FP16

MTP_QUANT_MODULES = [
    "mtp.layers.0.self_attn.q_proj",
    "mtp.layers.0.self_attn.k_proj",
    "mtp.layers.0.self_attn.v_proj",
    "mtp.layers.0.self_attn.o_proj",
    "mtp.layers.0.mlp.gate_proj",
    "mtp.layers.0.mlp.up_proj",
    "mtp.layers.0.mlp.down_proj",
]

assert os.path.exists(CKPT), f"Step 2 output not found: {CKPT}"
print("Paths OK.", flush=True)

In [ ]:
# ── Quantization helper
   
def quantize_group_sym_int8(w):
    """Symmetric per-group INT8. Returns (int8_q, bf16_scale)."""
    out, inp = w.shape
    assert inp % GROUP == 0, f"{inp} not divisible by group_size={GROUP}"
    ng   = inp // GROUP
    wg   = w.reshape(out, ng, GROUP).float()
    amax = wg.abs().amax(dim=2, keepdim=True)       # [out, ng, 1]
    scale = (amax / QMAX).clamp(min=1e-8)           # [out, ng, 1]
    q    = torch.round(wg / scale).clamp(-QMAX - 1, QMAX)
    q    = q.reshape(out, inp).to(torch.int8)
    scale = scale.reshape(out, ng).to(torch.bfloat16)
    return q, scale

In [ ]:
# ── Load checkpoint 

print("Loading checkpoint ...", flush=True)
tensors, meta = {}, None
with safe_open(CKPT, "pt") as f:
    meta = f.metadata()
    for k in f.keys():
        tensors[k] = f.get_tensor(k)
print(f"  {len(tensors)} tensors loaded", flush=True)

In [ ]:
# ── Quantize MTP Linear projections 

for mod in MTP_QUANT_MODULES:
    wkey = f"{mod}.weight"
    if wkey not in tensors:
        print(f"  !! {wkey} missing — check Step 2 ran correctly", flush=True)
        continue

    w = tensors.pop(wkey)        # remove FP16 original
    q, scale = quantize_group_sym_int8(w)

    packed = pack_to_int32(q.to(torch.int8), NB, packed_dim=1)
    tensors[f"{mod}.weight_packed"] = packed.contiguous()
    tensors[f"{mod}.weight_scale"]  = scale.contiguous()
    tensors[f"{mod}.weight_shape"]  = torch.tensor(list(w.shape), dtype=torch.int64)

    print(f"  ✓ {mod}: packed{tuple(packed.shape)}  scale{tuple(scale.shape)}", flush=True)

print(f"\nTotal tensors after MTP quant: {len(tensors)}", flush=True)

In [ ]:
# ── Save (in-place overwrite) 

print(f"Saving to {CKPT} ...", flush=True)
save_file(tensors, CKPT, metadata=meta or {"format": "pt"})
print(f"\n Step 3 DONE — MTP head quantized to INT8 pack-quantized W8A16", flush=True)

In [ ]:
# ── Verify 

with safe_open(CKPT, "pt") as f:
    all_keys = list(f.keys())

mtp_packed = [k for k in all_keys if k.startswith("mtp.") and k.endswith(".weight_packed")]
mtp_fp16   = [k for k in all_keys if k.startswith("mtp.") and k.endswith(".weight")]

print(f"MTP INT8 packed layers : {len(mtp_packed)}  (want 7)")
print(f"MTP FP16 .weight left  : {len(mtp_fp16)}   (mtp.fc + norms expected)")
print("MTP packed keys:", sorted(mtp_packed))